# B-Free x GlobalForge — Notebook 02: Kaggle Training (K2, LoRA r=16)

Huấn luyện **DINOv2 ViT-B/14 reg4 + LIB + GSR + L_DCS** (LoRA r=16, α=32) trên B-Free training data (51.517 real + 309.102 fake = 6 fake variants/ID), 504×504, 8 epochs, bf16.

**Locked decisions (plan.md D1–D10):**
- D1 `lambda_dcs=0.01`, `dcs_tau=0.07`, `label_smoothing=0.1`
- D2 8 epochs @ 504px — D3 full data (no subsample)
- D4 50/50 per-ID pairing: p=0.5 real else random 1/6 fake variant (resample mỗi epoch)
- D5 DINOv2 pretrained init **offline từ local weights** (không HF hub)
- D6 auto batch size (0.9× VRAM) — D7 bf16 autocast
- AdamW lr=1e-4 wd=1e-4 · CosineAnnealingLR eta_min=1e-7 per-step · grad clip 1.0
- Val split `md5(id)%100 < 3`; mỗi val ID = 1 real + 1 deterministic fake (balanced)

**Yêu cầu Kaggle inputs:**
- `bfree-wheels` — wheel bundle (như notebook 01)
- `bfree-training-data` — B-Free training dataset (tải từ grip.unina.it): `COCO_real_512/` + 6 thư mục `SD2.1_*/`, fake variant dùng **cùng tên file** với ảnh real tương ứng
- `dinov2-vitb14-reg4-pretrain` — weights `timm/vit_base_patch14_reg4_dinov2.lvd142m` (file `.safetensors` hoặc `.pth`) để init backbone offline

> Log riêng CE và DCS mỗi epoch (rule agent.md) → `train_log.csv` cho K4 / Phase 8.

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
import os, sys, glob

WHEELS_DIR = "/kaggle/input/bfree-wheels"
assert os.path.isdir(WHEELS_DIR), (
    f"Wheel bundle not found at {WHEELS_DIR}. "
    "Attach the 'bfree-wheels' Kaggle dataset before running this notebook."
)
wheels = sorted(glob.glob(os.path.join(WHEELS_DIR, "*.whl")))
print(f"Found {len(wheels)} wheels in {WHEELS_DIR}")
assert wheels, "No .whl files found in the wheel bundle."

!pip install --no-index --find-links={WHEELS_DIR} \
    torch==2.8.0+cu128 torchvision==0.23.0+cu128
!pip install --no-index --find-links={WHEELS_DIR} \
    timm==1.0.22 peft==0.15.2 transformers==4.55.4 \
    pandas==2.3.3 numpy==1.26.4 matplotlib==3.11.1 seaborn==0.13.2 \
    scikit-learn scipy pyyaml pillow tqdm safetensors

In [ ]:
REPO_URL = "https://github.com/P-Bao/B-Free.git"
REPO_DIR = "/kaggle/working/B-Free"
BRANCH = "integration/loss-backbone"

import os, sys, glob
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already cloned.")

assert os.path.isfile(os.path.join(REPO_DIR, "code", "networks", "bfree_globalforge_vit.py")), "Clone failed: backbone file missing."
stubs = glob.glob(os.path.join(REPO_DIR, "code", "modules", "*_stub.py"))
assert not stubs, f"Stub files still present (K0 not merged?): {stubs}"
print("OK: repo cloned on integration/loss-backbone, no stub files (K0 verified).")

In [ ]:
import sys

sys.path.insert(0, os.path.join(REPO_DIR, "code"))

import glob
import hashlib
import random

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from networks.bfree_globalforge_vit import BFreeGlobalForgeViT
from datasets.bfree_dataset import DegradationPipeline
from train_lora import apply_lora_to_backbone
from utils.normalization import get_list_norm
from utils.dmetrics import balanced_accuracy_score, roc_auc_score

DEVICE = "cuda:0"
print(f"torch={torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}, "
      f"{(getattr(torch.cuda.get_device_properties(0), 'total_mem', None) or getattr(torch.cuda.get_device_properties(0), 'total_memory')) / 1024**3:.1f} GB")

In [ ]:
# ============ Hyperparameters (locked D1-D10 — mọi giá trị ở đây, không hardcode chỗ khác) ============
CONFIG = {
    "arch": "vit_base_patch14_reg4_dinov2.lvd142m",
    "num_classes": 2,
    "img_size": 504,
    "pretrained": True,            # D5: DINOv2 pretrained init (loaded offline ở cell model)
    "lambda_dcs": 0.01,            # D1
    "dcs_tau": 0.07,
    "label_smoothing": 0.1,
    "lib_kernel": 3,
    "lib_tau": 0.5,
    "gsr_window": 3,
    "gsr_mask_prob": 1.0,
    "epochs": 8,                   # D2
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_targets": ["qkv", "proj", "fc1", "fc2"],   # GlobalForge convention (apply_lora_to_backbone)
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "max_grad_norm": 1.0,
    "scheduler_eta_min": 1e-7,
    "max_vram_frac": 0.9,           # D6
    "batch_hard_cap": 128,
    "val_md5_percentile": 3,        # md5(id)%100 < 3
    "val_fake_variant": "SD2.1_selfconditioned",
    "pair_real_prob": 0.5,          # D4
    "num_workers": 4,
    "seed": 42,
    "data_root": "/kaggle/input/bfree-training-data",
    "pretrain_weights_glob": "/kaggle/input/dinov2-vitb14-reg4-pretrain/*",
    "output_dir": "/kaggle/working",
}

DATA_ROOT = CONFIG["data_root"]
assert os.path.isdir(DATA_ROOT), (
    f"Training data not found at {DATA_ROOT}. Attach 'bfree-training-data' "
    "(download from https://www.grip.unina.it/download/prog/B-Free/training_data/)."
)

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])

REAL_DIR = os.path.join(DATA_ROOT, "COCO_real_512")
FAKE_DIRS = sorted(d for d in glob.glob(os.path.join(DATA_ROOT, "SD2.1_*")) if os.path.isdir(d))
assert os.path.isdir(REAL_DIR), f"COCO_real_512 missing under {DATA_ROOT}"
assert len(FAKE_DIRS) == 6, f"Expected 6 SD2.1_* variant dirs, found {len(FAKE_DIRS)}: {FAKE_DIRS}"
for d in FAKE_DIRS:
    print(f"  {os.path.basename(d)}: {len(glob.glob(os.path.join(d, '*')))} files")

### Dataset — 50/50 per-ID pairing (D4)

`BFreeDataset` của repo nhận CSV tĩnh (filename,label) nên không pairing động theo ID được; cell dưới reuses `DegradationPipeline` của repo và cài đúng convention:

- **Train**: 1 sample = 1 ID; `p=0.5` → real, ngược lại random 1/6 fake variant (resample mỗi `__getitem__` → 8 epochs phủ hết các variant).
- **Val** (`md5(stem)%100 < 3`): mỗi ID cho 2 samples — index chẵn = real, lẻ = deterministic fake `SD2.1_selfconditioned` (fallback: variant khác có sẵn) → balanced.
- Crop: train RandomCrop 504 (ảnh 512×512) + hflip 0.5; val center crop 504 — giống `BFreeDataset`.
- Degraded view (cho L_DCS): áp cho cả train lẫn val (matching `BFreeDataset` — degradation unconditional), pipeline GlobalForge JPEG 20-80 → blur k7 σ0.5-1.5 p0.8 → jitter p0.8.

In [ ]:
def build_stem_index(dirs):
    maps = []
    for d in dirs:
        m = {}
        for p in glob.glob(os.path.join(d, "*")):
            if os.path.isfile(p):
                m[os.path.splitext(os.path.basename(p))[0]] = p
        maps.append(m)
    return maps

REAL_MAP = build_stem_index([REAL_DIR])[0]
FAKE_MAPS = build_stem_index(FAKE_DIRS)
ALL_STEMS = sorted(REAL_MAP.keys())
print(f"real={len(REAL_MAP)}, fake variants={[len(m) for m in FAKE_MAPS]}")

def is_val_id(stem):
    return int(hashlib.md5(stem.encode()).hexdigest(), 16) % 100 < CONFIG["val_md5_percentile"]

TRAIN_IDS = [s for s in ALL_STEMS if not is_val_id(s)]
VAL_IDS = [s for s in ALL_STEMS if is_val_id(s)]
print(f"train IDs={len(TRAIN_IDS)}, val IDs={len(VAL_IDS)} (~{CONFIG['val_md5_percentile']}%)")

class PairDataset(Dataset):
    """Train: p=0.5 real else random 1/6 fake variant (resample mỗi epoch).
    Val: 2 samples/ID — chẵn=real, lẻ=deterministic fake variant (balanced)."""

    def __init__(self, ids, img_size, is_train, pair_real_prob, val_fake_variant):
        self.ids = ids
        self.img_size = img_size
        self.is_train = is_train
        self.pair_real_prob = pair_real_prob
        self.val_fake_variant = val_fake_variant
        self.normalize = T.Compose(get_list_norm("resnet"))
        self.degradation = DegradationPipeline()

    def __len__(self):
        return len(self.ids) * (1 if self.is_train else 2)

    def _pick_val_fake(self, stem):
        for m, d in zip(FAKE_MAPS, FAKE_DIRS):
            if os.path.basename(d) == self.val_fake_variant and stem in m:
                return m[stem]
        for m in FAKE_MAPS:
            if stem in m:
                return m[stem]
        return None

    def __getitem__(self, i):
        stem = self.ids[i if self.is_train else i // 2]
        if self.is_train:
            if random.random() < self.pair_real_prob or not any(stem in m for m in FAKE_MAPS):
                path, label = REAL_MAP[stem], 0
            else:
                maps = [m for m in FAKE_MAPS if stem in m]
                path = maps[random.randrange(len(maps))][stem]
                label = 1
        else:
            if i % 2 == 0:
                path, label = REAL_MAP[stem], 0
            else:
                path = self._pick_val_fake(stem)
                label = 1
        assert path is not None, f"No image found for id {stem}"
        img = Image.open(path).convert("RGB")

        if self.is_train:
            t, l, h, w = T.RandomCrop.get_params(img, (self.img_size, self.img_size))
            img = TF.crop(img, t, l, h, w)
            if random.random() > 0.5:
                img = TF.hflip(img)
        else:
            img = TF.center_crop(img, (self.img_size, self.img_size))

        img_deg = self.degradation(img)
        return self.normalize(img), self.normalize(img_deg), label

train_ds = PairDataset(TRAIN_IDS, CONFIG["img_size"], True, CONFIG["pair_real_prob"], CONFIG["val_fake_variant"])
val_ds = PairDataset(VAL_IDS, CONFIG["img_size"], False, CONFIG["pair_real_prob"], CONFIG["val_fake_variant"])

a, b, c = train_ds[0]
print(f"train_ds: {len(train_ds)} samples | clean {tuple(a.shape)}, deg {tuple(b.shape)}, label {c}")
a, b, c = val_ds[0]
_, _, c2 = val_ds[1]
print(f"val_ds  : {len(val_ds)} samples | labels {c},{c2} (expect 0,1)")
assert tuple(a.shape) == (3, CONFIG["img_size"], CONFIG["img_size"])

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(CONFIG["seed"])

### Model — BFreeGlobalForgeViT + LoRA r=16 (offline pretrained init)

- Backbone `timm vit_base_patch14_reg4_dinov2.lvd142m`, `set_input_size(504)`, `num_classes=2`.
- **Offline pretrained init (D5):** checkpoint DINOv2 mặc định là 518px (grid 37×37) → load vào model 504 (grid 36×36) cần **resample `pos_embed`** bằng `timm.layers.resample_abs_pos_embed` (đúng cách timm làm nội bộ). Head 2-class + LIB/GSR missing khi `strict=False` là behavior đúng.
- LoRA r=16 α=32 dropout 0.05 target `qkv/proj/fc1/fc2` qua `apply_lora_to_backbone` của repo; LIB + GSR + `fc_norm` + `head` vẫn full-train.

In [ ]:
import math

import torch


def load_pretrained_backbone(model, glob_pattern):
    """D5 offline: load DINOv2 pretrained weights (.safetensors/.pth) vào model.model.
    Resample pos_embed nếu grid checkpoint (thường 518px/37×37) khác grid model (504px/36×36).
    Filter key+shape (bỏ head 1000-class của ckpt nếu có) trước khi load strict=False."""
    matches = [m for m in sorted(glob.glob(glob_pattern)) if m.endswith((".safetensors", ".pth", ".pt"))]
    assert matches, f"No pretrained weights match {glob_pattern} — attach the dinov2 dataset."
    path = matches[0]
    print(f"Loading pretrained backbone from: {path}")

    if path.endswith(".safetensors"):
        from safetensors.torch import load_file
        sd = load_file(path)
    else:
        sd = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(sd, dict) and isinstance(sd.get("model"), dict):
        sd = sd["model"]
    sd = {k[len("model."):] if k.startswith("model.") else k: v for k, v in sd.items()}

    prefix = model.model.num_prefix_tokens
    if "pos_embed" in sd:
        pe = sd["pos_embed"]
        n_patch_old = pe.shape[1] - prefix
        old_hw = int(math.isqrt(n_patch_old))
        new_hw = model.model.patch_embed.grid_size[0]
        if old_hw != new_hw:
            from timm.layers.pos_embed import resample_abs_pos_embed
            print(f"resample pos_embed: {old_hw}x{old_hw} -> {new_hw}x{new_hw}")
            sd["pos_embed"] = resample_abs_pos_embed(
                pe, new_size=model.model.patch_embed.grid_size,
                old_size=(old_hw, old_hw), num_prefix_tokens=prefix,
            )

    ref = model.model.state_dict()
    dropped = [k for k, v in sd.items() if k not in ref or ref[k].shape != v.shape]
    if dropped:
        print(f"dropped {len(dropped)} incompatible keys (e.g. {dropped[:4]})")
    sd = {k: v for k, v in sd.items() if k not in dropped}

    report = model.model.load_state_dict(sd, strict=False)
    print(f"missing={len(report.missing_keys)} unexpected={len(report.unexpected_keys)}")
    if report.missing_keys:
        print("  missing:", report.missing_keys[:8])
    assert set(report.missing_keys) <= {"head.weight", "head.bias"}, (
        f"Backbone not fully initialized, missing: {report.missing_keys[:10]}")
    assert not report.unexpected_keys
    return model


model = BFreeGlobalForgeViT(
    arch=CONFIG["arch"], num_classes=CONFIG["num_classes"],
    img_size=CONFIG["img_size"], pretrained=False,
    use_lib=True, use_gsr=True, use_dcs=True,
    lib_kernel=CONFIG["lib_kernel"], lib_tau=CONFIG["lib_tau"],
    gsr_window=CONFIG["gsr_window"], gsr_mask_prob=CONFIG["gsr_mask_prob"],
    dcs_tau=CONFIG["dcs_tau"], lambda_dcs=CONFIG["lambda_dcs"],
    label_smoothing=CONFIG["label_smoothing"],
)
model = load_pretrained_backbone(model, CONFIG["pretrain_weights_glob"])
model = apply_lora_to_backbone(model, r=CONFIG["lora_rank"],
                               lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"])
model.to(DEVICE)

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"trainable params: {n_train:,} / {n_total:,} ({100 * n_train / n_total:.2f}%)")

### Auto batch size finder (D6)

Binary search batch size lớn nhất mà `compute_loss` (2 forwards: clean + degraded, bf16 autocast, backward) chạy được không OOM. Bắt đầu từ 2, doubling tới khi OOM hoặc hard cap, rồi binary search giữa khoảng đó.

In [ ]:
def try_batch(model, bs, img_size, device):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        x1 = torch.randn(bs, 3, img_size, img_size, device=device)
        x2 = torch.randn(bs, 3, img_size, img_size, device=device)
        y = torch.randint(0, 2, (bs,), device=device)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            total, _, _ = model.compute_loss(x1, x2, y)
        total.backward()
        del x1, x2, y, total
        return True
    except torch.cuda.OutOfMemoryError:
        return False
    finally:
        torch.cuda.empty_cache()

def find_max_batch_size(model, img_size, device, hard_cap):
    assert try_batch(model, 2, img_size, device), "Batch size 2 OOMs — environment problem."
    lo, bs, hi = 2, 4, None
    while bs <= hard_cap:
        if try_batch(model, bs, img_size, device):
            lo = bs
            bs *= 2
        else:
            hi = bs
            break
    if hi is None:
        print(f"[warn] no OOM up to hard cap {hard_cap}")
        hi = lo + 1
    while lo + 1 < hi:
        mid = (lo + hi) // 2
        if try_batch(model, mid, img_size, device):
            lo = mid
        else:
            hi = mid
    return lo

BATCH_SIZE = find_max_batch_size(model, CONFIG["img_size"], DEVICE, CONFIG["batch_hard_cap"])
CONFIG["batch_size"] = BATCH_SIZE
print(f"AUTO BATCH SIZE = {BATCH_SIZE}")

### Training loop — 8 epochs, bf16 autocast, grad clip 1.0

- `total = CE + lambda_dcs * DCS` qua `model.compute_loss` — log **riêng** CE và DCS.
- AdamW (lr 1e-4, wd 1e-4) trên trainable params; CosineAnnealingLR `T_max = epochs * len(train_loader)`, `eta_min=1e-7`, step mỗi batch.
- Lưu checkpoint **best theo val_bAcc** + checkpoint cuối; `train_log.csv` (epoch, train_loss, train_ce, train_dcs, val_loss, val_auc, val_bacc).

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=CONFIG["num_workers"], pin_memory=True, drop_last=True,
                         persistent_workers=True, worker_init_fn=seed_worker, generator=g)
val_loader = DataLoader(val_ds, batch_size=max(2, BATCH_SIZE // 2), shuffle=False,
                        num_workers=CONFIG["num_workers"], pin_memory=True)

optimizer = AdamW((p for p in model.parameters() if p.requires_grad),
                  lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"] * len(train_loader),
                              eta_min=CONFIG["scheduler_eta_min"])


def run_epoch(epoch):
    model.train()
    agg = {"total": 0.0, "ce": 0.0, "dcs": 0.0, "n": 0}
    for step, (img_clean, img_deg, labels) in enumerate(train_loader):
        img_clean = img_clean.to(DEVICE, non_blocking=True)
        img_deg = img_deg.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            total, ce, dcs = model.compute_loss(img_clean, img_deg, labels)
        total.backward()
        torch.nn.utils.clip_grad_norm_(
            (p for p in model.parameters() if p.requires_grad), CONFIG["max_grad_norm"])
        optimizer.step()
        scheduler.step()
        bs = labels.size(0)
        agg["total"] += float(total.detach()) * bs
        agg["ce"] += float(ce.detach()) * bs
        agg["dcs"] += float(dcs.detach()) * bs
        agg["n"] += bs
        if step % 20 == 0:
            print(f"ep{epoch} step {step}/{len(train_loader)} | "
                  f"total {agg['total']/max(agg['n'],1):.4f} (ce {agg['ce']/max(agg['n'],1):.4f}, "
                  f"dcs {agg['dcs']/max(agg['n'],1):.4f}) | lr {optimizer.param_groups[0]['lr']:.2e}",
                  flush=True)
    return {k: agg[k] / max(agg["n"], 1) for k in ("total", "ce", "dcs")}


@torch.no_grad()
def run_val():
    model.eval()
    agg = {"total": 0.0, "n": 0}
    scores, labels_all = [], []
    for img_clean, img_deg, labels in val_loader:
        img_clean = img_clean.to(DEVICE, non_blocking=True)
        img_deg = img_deg.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            total, _, _ = model.compute_loss(img_clean, img_deg, labels)
            out = model(img_clean)
        logits = out["logits"].float()
        scores.append((logits[:, 1] - logits[:, 0]).cpu())
        labels_all.append(labels.cpu())
        agg["total"] += float(total) * labels.size(0)
        agg["n"] += labels.size(0)
    scores = torch.cat(scores).numpy()
    labels_all = torch.cat(labels_all).numpy()
    return {
        "val_loss": agg["total"] / max(agg["n"], 1),
        "val_auc": roc_auc_score(labels_all, scores),
        "val_bacc": balanced_accuracy_score(labels_all, scores > 0),
    }


LOG_CSV = os.path.join(CONFIG["output_dir"], "train_log.csv")
with open(LOG_CSV, "w", encoding="utf-8") as f:
    f.write("epoch,train_loss,train_ce,train_dcs,val_loss,val_auc,val_bacc\n")

TRAIN_LOG = []
best_bacc = 0.0
for epoch in range(1, CONFIG["epochs"] + 1):
    tr = run_epoch(epoch)
    va = run_val()
    print(f"--> epoch {epoch}: train total={tr['total']:.4f} (ce={tr['ce']:.4f}, dcs={tr['dcs']:.4f}) | "
          f"val loss={va['val_loss']:.4f} auc={va['val_auc']:.4f} bacc={va['val_bacc']:.4f}", flush=True)
    row = {"epoch": epoch, "train_loss": tr["total"], "train_ce": tr["ce"],
           "train_dcs": tr["dcs"], **va}
    TRAIN_LOG.append(row)
    with open(LOG_CSV, "a", encoding="utf-8") as f:
        f.write(f"{epoch},{tr['total']:.4f},{tr['ce']:.4f},{tr['dcs']:.4f},"
                f"{va['val_loss']:.4f},{va['val_auc']:.4f},{va['val_bacc']:.4f}\n")
    if va["val_bacc"] > best_bacc:
        best_bacc = va["val_bacc"]
        torch.save({"model": model.state_dict(), "epoch": epoch, "val_bacc": best_bacc,
                    "config": CONFIG},
                   os.path.join(CONFIG["output_dir"], "bfree_globalforge_lora_r16_best.pth"))
        print(f"[*] best checkpoint saved (epoch {epoch}, bAcc {best_bacc:.4f})", flush=True)

torch.save({"model": model.state_dict(), "epoch": CONFIG["epochs"], "config": CONFIG,
            "train_log": TRAIN_LOG},
           os.path.join(CONFIG["output_dir"], "bfree_globalforge_lora_r16.pth"))
print("TRAINING COMPLETE")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(LOG_CSV)
display(df)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), dpi=120)
axes[0].plot(df["epoch"], df["train_loss"], marker="o", label="total")
axes[0].plot(df["epoch"], df["train_ce"], marker="s", label="CE")
axes[0].plot(df["epoch"], df["train_dcs"], marker="^", label="DCS")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("train loss"); axes[0].legend()
axes[0].set_title("Train losses (CE & DCS logged separately)")
axes[1].plot(df["epoch"], df["val_loss"], marker="o", color="tab:red")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("val loss"); axes[1].set_title("Val loss")
axes[2].plot(df["epoch"], df["val_auc"], marker="o", label="AUC")
axes[2].plot(df["epoch"], df["val_bacc"], marker="s", label="bAcc")
axes[2].set_xlabel("epoch"); axes[2].set_title("Val metrics"); axes[2].legend()
fig.tight_layout()
fig.savefig(os.path.join(CONFIG["output_dir"], "training_curves.png"))
print("saved training_curves.png")

## Training Complete (K2)

Outputs trong `/kaggle/working/`:
- `bfree_globalforge_lora_r16.pth` — checkpoint cuối (model + config + train_log)
- `bfree_globalforge_lora_r16_best.pth` — checkpoint best theo val_bAcc
- `train_log.csv` — CE/DCS log riêng từng epoch (dùng cho K4 / Phase 8)
- `training_curves.png`

**K2 checklist:** chạy hết 8 epochs không lỗi · checkpoint saved · training log CSV + curves plot.

**Tiếp theo:** `03_bfree_kaggle_eval.ipynb` (3 models × 17 wild subsets + standard benchmarks).